## MOLECULAR DESCRIPTORS CALCULATION (3D) - Eg with Mordred

#### 1. Set-up:

In [1]:
## packages to install if required (remove hashtag and run):
#!pip install pysmiles
#!pip install descriptastorus
#!pip install mordred


In [2]:
## import necessary libraries:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from pysmiles import read_smiles

from rdkit import Chem
#from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem, PandasTools

from mordred import Calculator, descriptors

from descriptastorus.descriptors.DescriptorGenerator import MakeGenerator


In [3]:
## export dataframe with substances + SMILES:
smiles_df = pd.read_csv('data/contaminants.csv', delimiter = ";")
smiles_df.head(5)


,name,substance_type,CAS_nb,SMILES,detected
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1


In [4]:
## data cleaning:
print(f"Length of database with NAs: {len(smiles_df.SMILES)}")

if smiles_df.SMILES.isna().sum() > 0:
    smiles_cleaned1 = smiles_df.loc[smiles_df.SMILES.notna()].copy()#smiles_df.dropna() ## remove NAs (cannot compute molecular descriptor if no SMILES structure present)
    print(f"Length of database without NAs (cleaned version): {len(smiles_cleaned1.SMILES)}")
else:
    None


Length of database with NAs: 70
Length of database without NAs (cleaned version): 67


In [5]:
## extract list of molecules from their SMILES:
smiles_list = smiles_cleaned1.SMILES

mol_list = []

for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    mol_list.append(mol)


In [6]:
## append to df + rename columns with mols to 'molecule'
smiles_cleaned = pd.concat([smiles_cleaned1.reset_index(drop = True), pd.DataFrame(mol_list).reset_index(drop = True)], axis = 1)
smiles_cleaned = smiles_cleaned.rename(columns = {0: "molecule"})
smiles_cleaned


,name,substance_type,CAS_nb,SMILES,detected,molecule
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17e5584a0>
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e559000>
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17e558dd0>
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f20>
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f90>
...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a650>
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a6c0>
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a730>
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a7a0>


#### 2. Calculation of molecular descriptors:

In [7]:
## set-up Mordred calculator:
calculator_mordred = Calculator(descriptors, ignore_3D = True)


In [9]:
%%capture
## calculate molecular descriptors:
md_df = calculator_mordred.pandas(mol_list)


In [10]:
## reframe df (error on last line => bug fixed)
df = pd.concat([smiles_cleaned.reset_index(drop = True), pd.DataFrame(md_df).reset_index(drop = True)], axis = 1)
df


,name,substance_type,CAS_nb,SMILES,detected,molecule,ABC,ABCGG,nAcid,nBase,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17e5584a0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444,3.472222
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e559000>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944,4.180556
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17e558dd0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389,6.166667
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f20>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667,4.000000
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f90>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444,4.229167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a650>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,...,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944,4.291667
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a6c0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,1,0,...,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111,1.333333
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a730>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111,1.000000
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a7a0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.0,1.000000


#### 3. Cleaning of MD full of NAs (remove these columns):

In [11]:
mordred_md = pd.concat([smiles_cleaned, pd.DataFrame(md_df)], axis = 1)
mordred_md


,name,substance_type,CAS_nb,SMILES,detected,molecule,ABC,ABCGG,nAcid,nBase,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17e5584a0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444,3.472222
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e559000>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944,4.180556
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17e558dd0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389,6.166667
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f20>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667,4.000000
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f90>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444,4.229167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a650>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,...,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944,4.291667
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a6c0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,1,0,...,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111,1.333333
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a730>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111,1.000000
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a7a0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.0,1.000000


In [12]:
## convert df back to numeric (so descriptors with errors can be converted to NAs & then later removed):
mordred_md_num = mordred_md.iloc[:, 5:-1].apply(pd.to_numeric, errors = "coerce")

mordred_md_clean = mordred_md_num.dropna(axis = 1, how = "all")
mordred_md_clean


,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,VE1_A,VE2_A,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1
0,0,0,19.544560,2.377815,4.755629,19.544560,1.302971,3.627536,3.457764,0.230518,...,0.000000,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444
1,0,0,20.971017,2.377461,4.754921,20.971017,1.165057,3.775189,3.759106,0.208839,...,0.000000,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944
2,0,0,34.992970,2.613976,4.943050,34.992970,1.249749,4.277221,3.188891,0.113889,...,8.444838,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389
3,0,0,22.540937,2.377662,4.755325,22.540937,1.252274,3.804096,3.839997,0.213333,...,0.000000,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667
4,0,0,24.107183,2.519762,5.039524,24.107183,1.205359,3.919405,3.804569,0.190228,...,0.000000,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,0,1,24.017666,2.462551,4.925102,24.017666,1.264088,3.859038,3.738528,0.196765,...,0.000000,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944
63,1,0,5.226252,1.847759,3.695518,5.226252,1.045250,2.408576,2.130986,0.426197,...,0.000000,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111
64,0,0,3.464102,1.732051,3.464102,3.464102,0.866025,2.178059,1.931852,0.482963,...,0.000000,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111
65,0,0,2.000000,1.000000,2.000000,2.000000,1.000000,1.407606,1.414214,0.707107,...,0.000000,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.000000


In [13]:
## append MD to original df (now that MD are cleaned => rows full of NAs removed):
mordred_df = pd.concat([smiles_cleaned, mordred_md_clean], axis = 1)
mordred_df


,name,substance_type,CAS_nb,SMILES,detected,molecule,nAcid,nBase,SpAbs_A,SpMax_A,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17e5584a0>,0,0,19.544560,2.377815,...,0.000000,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e559000>,0,0,20.971017,2.377461,...,0.000000,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17e558dd0>,0,0,34.992970,2.613976,...,8.444838,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f20>,0,0,22.540937,2.377662,...,0.000000,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17e558f90>,0,0,24.107183,2.519762,...,0.000000,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a650>,0,1,24.017666,2.462551,...,0.000000,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a6c0>,1,0,5.226252,1.847759,...,0.000000,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a730>,0,0,3.464102,1.732051,...,0.000000,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17e55a7a0>,0,0,2.000000,1.000000,...,0.000000,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.000000


#### 4. Export results to .csv file:

In [14]:
mordred_df.to_csv("data/mordred_descriptors.csv", index = False)
